# Notebook 4 — GCCM Publication Figures

Generates the Geographical Convergent Cross-Mapping (GCCM) figures for the
micropublication:

1. **Convergence plot** — faceted line plot showing cross-map skill (ρ) vs
   library size for each predictor, with 95% CI ribbons.
2. **Directional asymmetry plot** — horizontal bar chart comparing forward
   (pred → LST) and reverse (LST → pred) cross-map skill at the largest
   library size.

**Inputs:** Pre-computed GCCM results in `outputs/gccm/main_E3_tau1/`.
These are produced by `R/gccm_analysis.R`.

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

## Configuration

In [ ]:
# -- Paths -----------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "outputs" / "gccm" / "main_E3_tau1"
FIG_DIR = PROJECT_ROOT / "figures" / "pub"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# -- Colors — SHAP palette -------------------------------------------------
COLORS = {
    "forward": "#008afa",   # pred → LST (SHAP blue)
    "reverse": "#ff0051",   # LST → pred (SHAP red)
    "positive": "#008afa",  # positive asymmetry
    "negative": "#ff0051",  # negative asymmetry
    "axis": "#333333",
    "tick_label": "#999999",
    "hline": "#cccccc",
}

# -- Font sizes — scaled 2x for compositing at ~1/4 page height ------------
FONT = {
    "feature_name": 22,
    "axis_label": 22,
    "tick": 18,
    "legend": 18,
    "title": 24,
    "annotation": 16,
}

DPI = 300

In [ ]:
# -- Apply SHAP-matching style ---------------------------------------------
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": FONT["tick"],
    "axes.titlesize": FONT["title"],
    "axes.titleweight": "bold",
    "axes.labelsize": FONT["axis_label"],
    "axes.labelcolor": COLORS["axis"],
    "axes.linewidth": 0.8,
    "axes.edgecolor": COLORS["axis"],
    "axes.grid": False,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.facecolor": "white",
    "xtick.labelsize": FONT["tick"],
    "ytick.labelsize": FONT["tick"],
    "xtick.color": COLORS["axis"],
    "ytick.color": COLORS["axis"],
    "legend.fontsize": FONT["legend"],
    "legend.framealpha": 0.9,
    "legend.edgecolor": "0.8",
    "figure.facecolor": "white",
    "figure.dpi": DPI,
    "savefig.dpi": DPI,
    "savefig.bbox": "tight",
    "savefig.facecolor": "white",
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
})

## Load data

In [ ]:
results = pd.read_csv(DATA_DIR / "results.csv")
summary = pd.read_csv(DATA_DIR / "summary.csv")

print(f"Results: {results.shape[0]} rows, {results['predictor'].nunique()} predictors")
print(f"Summary: {summary.shape[0]} rows")
summary[["predictor", "group", "rho_pred_to_LST", "rho_LST_to_pred", "direction"]]

## Figure 1: GCCM Convergence

In [ ]:
# Panel layout: causal predictors on top row, associative on bottom
PANEL_ORDER = [
    "built_density", "green_density", "distance_to_roads", "distance_to_water",
    "DEM", "NDVI", "NDBI", "BSI",
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8), sharex=True, sharey=True)
fig.suptitle("GCCM Convergence", fontsize=FONT["title"], fontweight="bold", y=1.02)

for ax, pred in zip(axes.flat, PANEL_ORDER):
    df_pred = results[results["predictor"] == pred]

    # Forward direction: pred → LST
    fwd = df_pred[df_pred["is_pred_to_lst"] == True].sort_values("libsize")
    ax.plot(fwd["libsize"], fwd["rho"], color=COLORS["forward"],
            marker="o", markersize=4, label="pred → LST")
    ax.fill_between(fwd["libsize"], fwd["ci_lower"], fwd["ci_upper"],
                    color=COLORS["forward"], alpha=0.15)

    # Reverse direction: LST → pred
    rev = df_pred[df_pred["is_pred_to_lst"] == False].sort_values("libsize")
    ax.plot(rev["libsize"], rev["rho"], color=COLORS["reverse"],
            marker="s", markersize=4, linestyle="--", label="LST → pred")
    ax.fill_between(rev["libsize"], rev["ci_lower"], rev["ci_upper"],
                    color=COLORS["reverse"], alpha=0.15)

    ax.set_title(pred, fontsize=FONT["feature_name"], fontweight="bold")
    ax.tick_params(axis="y", length=0)

# Shared axis labels
for ax in axes[1, :]:
    ax.set_xlabel("Library size (pixels)", fontsize=FONT["axis_label"])
for ax in axes[:, 0]:
    ax.set_ylabel(r"Cross-map skill ($\rho$)", fontsize=FONT["axis_label"])

# Single shared legend
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2,
           fontsize=FONT["legend"], bbox_to_anchor=(0.5, -0.04))

fig.tight_layout()
fig.savefig(FIG_DIR / "gccm_convergence_tau1.png", bbox_inches="tight")
print(f"Saved: {FIG_DIR / 'gccm_convergence_tau1.png'}")
plt.show()

## Figure 2: GCCM Directional Asymmetry

In [ ]:
# Compute asymmetry = forward - reverse (at largest library size)
asym = summary[["predictor", "rho_pred_to_LST", "rho_LST_to_pred"]].copy()
asym["asymmetry"] = asym["rho_pred_to_LST"] - asym["rho_LST_to_pred"]

# Sort: causal predictors (positive asymmetry) first, then associative
asym = asym.sort_values("asymmetry", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(asym))
bar_height = 0.35

# Forward bars (pred → LST)
ax.barh([y + bar_height / 2 for y in y_pos], asym["rho_pred_to_LST"],
        height=bar_height, color=COLORS["forward"], label="pred → LST (forward)")
# Reverse bars (LST → pred)
ax.barh([y - bar_height / 2 for y in y_pos], asym["rho_LST_to_pred"],
        height=bar_height, color=COLORS["reverse"], label="LST → pred (reverse)")

# Asymmetry annotations
for i, (_, row) in enumerate(asym.iterrows()):
    val = row["asymmetry"]
    max_rho = max(row["rho_pred_to_LST"], row["rho_LST_to_pred"])
    color = COLORS["positive"] if val > 0 else COLORS["negative"]
    ax.text(max_rho + 0.015, i, f"{val:+.3f}",
            va="center", fontsize=FONT["annotation"],
            fontweight="bold", color=color)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(asym["predictor"], fontsize=FONT["feature_name"])
ax.set_xlabel(r"Cross-mapping skill ($\rho$)", fontsize=FONT["axis_label"])
ax.set_title("GCCM Directional Asymmetry",
             fontsize=FONT["title"], fontweight="bold")
ax.legend(fontsize=FONT["legend"], loc="upper right")
ax.tick_params(axis="y", length=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / "gccm_asymmetry_tau1.png", bbox_inches="tight")
print(f"Saved: {FIG_DIR / 'gccm_asymmetry_tau1.png'}")
plt.show()